# FinChain 06 - Distributed training lab

This notebook is intentionally separate from the main RLVR training notebooks.

The purpose is to learn distributed vocabulary and failure modes: rank, world size, process groups, distributed samplers, DDP, FSDP, ZeRO, sharded checkpoints, and when rollout parallelism is enough.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import platform
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/workspace/finpost") if Path("/workspace/finpost").exists() else PROJECT_ROOT

RESULTS_DIR = PROJECT_ROOT / "results" / "finchain_rlvr"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def progress(title, detail=None):
    stamp = time.strftime("%H:%M:%S")
    print(f"[{stamp}] {title}")
    if detail:
        print(detail)

def check_module(name):
    try:
        return importlib.import_module(name), None
    except Exception as exc:
        return None, exc

def run_cmd(cmd, *, check=False):
    progress("running command", cmd)
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed with exit {completed.returncode}: {cmd}")
    return completed

def append_cost_event(stage, **payload):
    path = RESULTS_DIR / "cost_ledger.jsonl"
    row = {"stage": stage, "time": time.strftime("%Y-%m-%dT%H:%M:%S"), **payload}
    with path.open("a", encoding="utf-8") as fp:
        fp.write(json.dumps(row, sort_keys=True) + "\n")
    print(json.dumps(row, indent=2, sort_keys=True))
    return row

progress("project root", str(PROJECT_ROOT))
progress("python", sys.version.split()[0])
progress("platform", platform.platform())


In [ ]:
progress("GPU preflight")
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"gpu {idx}: {props.name}, vram={props.total_memory / 1e9:.1f} GB")
except Exception as exc:
    print("torch check failed:", repr(exc))

run_cmd("nvidia-smi")


In [ ]:
progress("distributed environment variables")
for key in ["RANK", "LOCAL_RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT", "CUDA_VISIBLE_DEVICES"]:
    print(f"{key}={os.environ.get(key)}")


In [ ]:
progress("sampler intuition without launching distributed processes")
examples = list(range(12))
world_size = 3
for rank in range(world_size):
    shard = examples[rank::world_size]
    print(f"rank {rank} sees {shard}")


In [ ]:
progress("when to use which scaling mode")
rows = [
    ("parallel rollouts", "sampling throughput bottleneck", "first multi-GPU win for OPD/GRPO"),
    ("DDP", "model fits on each GPU", "throughput and global batch scaling"),
    ("FSDP", "model/optimizer state does not fit", "state sharding and sharded checkpoints"),
    ("tensor parallel", "single layer/model too large", "mostly out of scope for 3B/4B LoRA"),
]
for row in rows:
    print(f"{row[0]:18} | {row[1]:36} | {row[2]}")


In [ ]:
append_cost_event(
    stage="notebook_preflight",
    notebook=Path(globals().get("__vsc_ipynb_file__", "unknown")).name,
    gpu_count=0,
    notes="preflight cell executed; update this ledger in each expensive stage",
)


## Exit gate

- You can explain rank and world size.
- You can explain why DDP does not reduce model memory.
- You can explain why FSDP/ZeRO reduce per-GPU training state.
- You know why rollout parallelism may be more useful than distributed training for the first RLVR scaling experiment.
